In [27]:
from src.cv_model import run_ablation_experiment,run_xgb_gridsearch,train_final_model
from src.utils.data_processing import load_data_base,format_features,format_features, add_demog_features
import pandas as pd

In [28]:
dossier="/home/tiphainell/Documents/5.Direct Assurance/actuarial-loss-estimation"
database=load_data_base(dossier)
#format database
database=format_features(database)



In [29]:
#add demographic_features
database=add_demog_features(database)


list_features_demog=database.columns.to_list()

In [30]:
list_features_demog

cols_to_remove = [
    "ClaimNumber",
    "DateTimeOfAccident",
    "UltimateIncurredClaimCost",
    "DateReported",
    "ClaimDescription",
    "ClaimDescriptionClean",
    "role",
    "claim_development_ratio"
]

features_demog_x = [
    col for col in list_features_demog
    if col not in cols_to_remove
]

features_demog_x

['Age',
 'DependentChildren',
 'DependentsOther',
 'WeeklyWages',
 'HoursWorkedPerWeek',
 'DaysWorkedPerWeek',
 'InitialIncurredCalimsCost',
 'Gender_M',
 'Gender_U',
 'MaritalStatus_S',
 'MaritalStatus_U',
 'PartTimeFullTime_P',
 'DateTimeOfAccident_year',
 'DateTimeOfAccident_month',
 'DateTimeOfAccident_day',
 'DateTimeOfAccident_hour',
 'DateTimeOfAccident_week',
 'reporting_delay',
 'wages_per_hour',
 'claim_given_wages',
 'age_2']

In [31]:
train=database[database["role"]=="train"]
model=train_final_model(train,features_demog_x,"UltimateIncurredClaimCost")

In [36]:
import numpy as np
test=database[database['role']=="test"]
X_test=test[features_demog_x]
y_pred=np.expm1(model.predict(X_test))

In [37]:
results=pd.DataFrame({
   "ClaimNumber":test["ClaimNumber"],
   "UltimateIncurredClaimCost":y_pred})
results

,ClaimNumber,UltimateIncurredClaimCost
0,WC8145235,7096.963867
1,WC2005111,2782.041504
2,WC6899143,24085.136719
3,WC5502023,422.137054
4,WC4785156,3794.250244
...,...,...
35995,WC9666858,10862.680664
35996,WC4800526,691.939148
35997,WC3360567,6464.019531
35998,WC7491778,6708.414062


In [38]:
results.to_csv("final_model.csv", index=False)